Question 4 : How are discussions about coding influenced by trends in remote work, digital nomadism, and gig economy employment, from the point of views of coding professionals?

Approach:

1 - Twitter 
2 - Dev.To (THIS NOTEBOOK)


In [1]:
#Imports
import requests
from bs4 import BeautifulSoup
from collections import Counter
import spacy
import pickle
import json

To get the articles from Dev.To, I used their API to do a query on articles. Even though the GET query accepts params such as 'tags' and 'query' that I have used here they're not as accurate in filtering (this is still beta version of API), so I did a double check in the end before saving the results(Title & URL of article) to the file.

In [2]:
with open ("keywordsAndSubFiltered.json") as f:
    keywordsAndSubFiltered = (json.load (f))

remoteWorkKeywordsList = []
for k, v in keywordsAndSubFiltered.items():
    remoteWorkKeywordsList.append(k)
    remoteWorkKeywordsList.extend(v)

In [3]:
with open ("Jobtitles.pickle", "rb") as f:
    jobTitles = pickle.load(f)

In [4]:
api_key = '37FgdNeqhYLvCFK96Uj7wGZ1'

# Endpoint URL for fetching articles
url = 'https://dev.to/api/articles'

# Query parameters to filter articles based on search query
params = {
    'tags': remoteWorkKeywordsList,
    'q': jobTitles,
    'per_page': '1000',
    'api_key': api_key 
}

# Make a GET request to fetch articles
response = requests.get(url, params=params)

# Check if the request was successful and the criteria are met
if response.status_code == 200:
    articles = response.json()
    print(len(articles))
    # Iterate over retrieved articles and get relevant information
    with open("urlsDevTo.txt", 'w') as file:
        for article in articles:
            article_tags = article.get('tags', [])
            title = article['title']
            body = article['description']
            # Double check
            if any(phrase in title.lower() or phrase in body.lower() for phrase in jobTitles) and any(tag in article_tags.lower() for tag in remoteWorkKeywordsList):
                title = article['title']
                url = article['url']
                file.write(f"Title: {title}\nURL: {url}\n")
else:
    print(f"Failed to fetch articles. Status code: {response.status_code}")


Failed to fetch articles. Status code: 502


After having a file containing the matching articles, the next step is to determine the most frequent of the emerging technologies as described in the articles.
The following function gets the articles from the "urlsDevTo.txt".

In [2]:
def read_articles(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    articles = []
    for i in range(0, len(lines), 2):
        title = lines[i].strip().replace('Title: ', '')
        url = lines[i+1].strip().replace('URL: ', '')
        articles.append((title, url))
    return articles

IndexError: list index out of range

The following function scrapes the website Dev.To to get the body (content) of the  articles.

In [11]:
def fetch_article_body(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        article_body = soup.find(class_='crayons-article__body')
        return article_body.get_text(strip=True) if article_body else None
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

Here I'm using spaCy which is a library for natural language processing (NLP) in Python, to analyze the contents of the articles and extract potential technology-related keywords. 

In [12]:
nlp = spacy.load('en_core_web_sm')

def analyze_technologies(text):
    doc = nlp(text)
    tech_keywords = Counter()

    # Here you can choose to focus on NOUNs, PROPNs (proper nouns), or ENTs (entities)
    for token in doc:
        # Filtering for proper nouns might help narrow down to names of technologies
        if token.pos_ == 'PROPN':
            tech_keywords[token.text] += 1

    # Or you can focus on specific types of named entities, such as 'ORG'
    for ent in doc.ents:
        if ent.label_ == 'ORG':
            tech_keywords[ent.text] += 1

    return tech_keywords


The following is the "main" code that reads the articles, initializes the counter and sets the frequency for the most important technologies mentioned and saves it in the 'discoveryDevTo.txt' file.

In [13]:
file_path = 'urlsDevTo.txt' 
articles = read_articles(file_path)
global_counter = Counter()

for title, url in articles:
    body = fetch_article_body(url)
    if body:
        tech_counts = analyze_technologies(body)
        global_counter.update(tech_counts)

# Save the output to a file
    with open('discoveryDevTo.txt', 'w') as file:
        for tech, count in global_counter.most_common(25):
            file.write(f"{tech}: {count}\n")